In [17]:
import torch
import numpy as np
import torchvision
import torch.nn.functional as F
from PIL import Image, ImageOps
from torchvision import transforms
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sklearn.decomposition import PCA


print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

PyTorch: 2.9.1
Torchvision: 0.24.1


In [19]:
base_dir = (
    Path.home()
    / "Documents"
    / "Toronto Graduate Study"
    / "MASc Thesis"
    / "World_Model"
    / "3D_awareness_in_2D"
)

REPO_DIR = base_dir / "dinov3-main" 

vits16_weight_path = REPO_DIR /  "model_checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"
vits16plus_weight_path = REPO_DIR / 'model_checkpoints/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth'
vitb16_weight_path = REPO_DIR / 'model_checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'

# DINOv3 ViT models pretrained on web images
dinov3_vits16 = torch.hub.load(REPO_DIR, 'dinov3_vits16', source='local', weights=str(vits16_weight_path))
dinov3_vits16plus = torch.hub.load(REPO_DIR, 'dinov3_vits16plus', source='local', weights=str(vits16plus_weight_path))
dinov3_vitb16 = torch.hub.load(REPO_DIR, 'dinov3_vitb16', source='local', weights=str(vitb16_weight_path))

In [22]:
# baseline 
def extract_patch_tokens(image, model, patch_size=16, max_size=512):
    """
    Extract normalized DINOv3 patch tokens from a PIL image.

    Returns
    -------
    processed_image : PIL.Image
    patch_tokens    : Tensor [N, D]
    rows            : int
    cols            : int
    """
    model.eval()

    # Fix image orientation and ensure RGB
    processed_image = ImageOps.exif_transpose(image).convert("RGB")

    # Optional: keep image manageable while preserving aspect ratio
    processed_image.thumbnail(
        (max_size, max_size),
        Image.Resampling.LANCZOS
    )

    width, height = processed_image.size

    # Crop so dimensions are exact multiples of patch size
    cols = width // patch_size
    rows = height // patch_size

    if rows == 0 or cols == 0:
        raise ValueError(
            f"Image must be at least {patch_size} x {patch_size} pixels."
        )

    processed_image = processed_image.crop(
        (0, 0, cols * patch_size, rows * patch_size)
    )

    # Prepare the model input
    # image transforms as in dinov3 github
    # we need to normalize as the image distribution should stay
    # the same as the image distribution encountered during dinov3 training
    preprocess = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])

    parameter = next(model.parameters())

    x = preprocess(processed_image).unsqueeze(0).to(
        device = parameter.device,
        dtype = parameter.dtype
    )

    with torch.inference_mode():
        features = model.forward_features(x)

        # DINOv3:
        # shape [B, N_patches, D]
        patch_tokens = features["x_norm_patchtokens"][0]

    # Normalize explicitly so dot product = cosine similarity
    patch_tokens = F.normalize(
        patch_tokens.float(),
        p=2,
        dim=-1
    )

    assert patch_tokens.shape[0] == rows * cols, (
        f"Expected {rows * cols} patch tokens, "
        f"but got {patch_tokens.shape[0]}"
    )

    return processed_image, patch_tokens, rows, cols

In [21]:
def cosine_sim_match(
    query_patch,
    image_1,
    image_2,
    model,
    patch_size=16,
):
    """
    Match one query patch from image_1 to the most similar patch in image_2
    using cosine similarity of DINOv3 patch tokens.

    Parameters
    ----------
    query_patch : tuple
        (row, col) location of the query patch in image_1's patch grid.

    image_1, image_2 : PIL.Image
        Two images/views.

    model : torch.nn.Module
        Frozen DINOv3 model.

    patch_size : int
        ViT patch size. DINOv3 ViT-S/16 -> 16.

    Returns
    -------
    Dictionary containing:
        query_patch
        matched_patch
        max_similarity
        similarity_map
        processed images
    """

    # ---------------------------------------------------
    # 1. Extract patch features from both images
    # ---------------------------------------------------

    img1, tokens1, rows1, cols1 = extract_patch_tokens(
        image_1,
        model,
        patch_size=patch_size,
    )

    img2, tokens2, rows2, cols2 = extract_patch_tokens(
        image_2,
        model,
        patch_size=patch_size,
    )

    # ---------------------------------------------------
    # 2. Convert query (row, col) to flattened token index
    # ---------------------------------------------------

    query_row, query_col = query_patch

    if not (0 <= query_row < rows1):
        raise ValueError(
            f"query_row must be in [0, {rows1 - 1}]"
        )

    if not (0 <= query_col < cols1):
        raise ValueError(
            f"query_col must be in [0, {cols1 - 1}]"
        )

    query_index = query_row * cols1 + query_col

    # [D]
    query_feature = tokens1[query_index]

    # ---------------------------------------------------
    # 3. Compare query against EVERY patch in image 2
    # ---------------------------------------------------

    # tokens2:       [N2, D]
    # query_feature: [D]
    #
    # Since both are normalized:
    #
    # tokens2 @ query_feature
    #
    # is cosine similarity.
    similarities = tokens2 @ query_feature

    # ---------------------------------------------------
    # 4. Find nearest patch
    # ---------------------------------------------------

    best_index = similarities.argmax().item()

    matched_row = best_index // cols2
    matched_col = best_index % cols2

    max_similarity = similarities[best_index].item()

    # ---------------------------------------------------
    # 5. Reshape similarities to spatial image grid
    # ---------------------------------------------------

    similarity_map = similarities.reshape(rows2, cols2)

    return {
        "query_patch": (query_row, query_col),
        "query_index": query_index,

        "matched_patch": (matched_row, matched_col),
        "matched_index": best_index,

        "max_similarity": max_similarity,
        "similarity_map": similarity_map.cpu(),

        "image_1": img1,
        "image_2": img2,

        "grid_1": (rows1, cols1),
        "grid_2": (rows2, cols2),
    }